In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Running this notebook on: ", device)

import spateo as st
print("Last run with spateo version:", st.__version__)

# Other imports
import warnings, string
import anndata as ad
import dynamo as dyn
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Uncomment the following if running on the server
import pyvista as pv
pv.start_xvfb()

sns.set_theme(context="paper", style="ticks", font_scale=1)
warnings.filterwarnings('ignore')
# %load_ext autoreload

Running this notebook on:  cpu


2025-06-14 12:35:54.283984: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/root/miniforge3/envs/spateo/lib/python3.8/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


fastpd is not installed. If you need mesh correction, please compile the fastpd library.
Last run with spateo version: 0.0.0


In [2]:
adata = sc.read_h5ad('/data/work/05.cluster/FuseMap/0106/thalamus_latent_embeddings_all_spatial_pretrain/dmt_leiden_20250108_1.h5ad')
adata

AnnData object with n_obs × n_vars = 3279854 × 33347
    obs: 'dnbCount', 'area', 'orig.ident', 'x', 'y', 'region', 'n_counts', 'region_h2', 'slice_code', 'sub_region', 'dmt_leiden', 'annotation_1230', 'dmt_leiden_merge'
    uns: 'dmt_leiden_colors', 'dmt_leiden_merge_colors', 'dmt_nn', 'leiden', 'slice_code_colors'
    obsm: 'X_dmt', 'X_dmt_highdim', 'align_spatial_2d', 'align_spatial_3d', 'cell_border', 'latent_embeddings_all_single_pretrain', 'latent_embeddings_all_spatial_pretrain', 'spatial', 'spatial_division'
    obsp: 'dmt_nn_connectivities', 'dmt_nn_distances'

In [3]:
dic = {'0': 'T7_SST_LHX_SIX3',
 '1': 'T9_CRH',
 '10': 'T6_CLSTN2',
 '11': 'T4_COL4A1_vessel',
 '12': 'T3_BMP4_IGFBP5',
 '13': 'T7_SST_LHX_SIX3',
 '14': 'T0_PCP4_VGF',
 '15': 'T9_CRH',
 '16': 'T3_BMP4_IGFBP5',
 '17': 'T3_BMP4_IGFBP5',
 '18': 'T2_PRSS12',
 '19': 'T9_CRH',
 '2': 'T9_CRH',
 '20': 'T1_NTS_CALB1',
 '21': 'T1_NTS_CALB1',
 '22': 'T1_NTS_CALB1',
 '23': 'T10_HBZ',
 '24': 'T1_NTS_CALB1',
 '25': 'T1_NTS_CALB1',
 '26': 'T2_PRSS12',
 '27': 'T9_CRH',
 '28': 'T2_PRSS12',
 '29': 'T5_AQP4_gial_region',
 '3': 'T10_HBZ',
 '30': 'T8_CBLN4_CBLN1',
 '31': 'T7_SST_LHX_SIX3',
 '32': 'T9_CRH',
 '33': 'T1_NTS_CALB1',
 '34': 'T7_SST_LHX_SIX3',
 '35': 'T10_HBZ',
 '36': 'T8_CBLN4_CBLN1',
 '37': 'T9_CRH',
 '38': 'T3_BMP4_IGFBP5',
 '39': 'T1_NTS_CALB1',
 '4': 'T2_PRSS12',
 '40': 'T9_CRH',
 '41': 'T6_CLSTN2',
 '42': 'T10_HBZ',
 '5': 'T7_SST_LHX_SIX3',
 '6': 'T7_SST_LHX_SIX3',
 '7': 'T9_CRH',
 '8': 'T2_PRSS12',
 '9': 'T1_NTS_CALB1'}

adata.obs['dmt_leiden_annotation_0115'] = [dic[i] for i in adata.obs['dmt_leiden']]
colormap = {'T2_PRSS12': '#31d6d3',
 'T6_CLSTN2': '#6b0c4d',
 'T3_BMP4_IGFBP5': '#e94a1d',
 'T4_COL4A1_vessel': '#cf58e5',
 'T10_HBZ': '#39d789',
 'T7_SST_LHX_SIX3': '#4e7c26',
 'T8_CBLN4_CBLN1': '#eaccd8',
 'T1_NTS_CALB1': '#9114fb',
 'T9_CRH': '#79f4ec',
 'T0_PCP4_VGF': '#b3fcdd',
 'T5_AQP4_gial_region': '#455edf'}

In [4]:
names = [
 '14_A03591A1C3_WT202403310045.h5ad',
 '16_A03592A4C6_WT202403310044.h5ad',
 '18_B03602C4D6_WT202405020031.h5ad',
 '20_B03606F3G5_WT202405020032.h5ad',
 '22_B03606C4E6_WT202403310050.h5ad',
 '23_B03609A4D6_WT202404150263.h5ad',
 '27_B03610C1E3_WT202403310051.h5ad',
 '31_B03619A1D3_WT202403310052.h5ad',
 '35_B03619E4G6_WT202403310053.h5ad',
 '39_A03589A1D4_WT202403310046.h5ad',
 '43_A03590E1G4_WT202403310064.h5ad',
 '47_A03593C1F3_WT202403310068.h5ad',
 '51_B03605C2E5_WT202406020126.h5ad',
 '55_B03613E3G6_WT202403310069.h5ad',
 '59_B03612E4G6_WT202403310059.h5ad',
 '63_B03606C1E3_WT202403310061.h5ad',
 '67_A03595A1D3_WT202403310062.h5ad',
 '71_A03595A4D6_WT202403310063.h5ad',
 '75_D03468D1E3_WT202403310066.h5ad',
 '80_D03473D4E6_WT202403310070.h5ad',
 '84_B03423D1E3_WT202403310065.h5ad',
 '89_D03466A1C3_WT202403310058.h5ad',
 '99_D03470D1E3_WT202404290071.h5ad',
 '104_B03615F3G5_WT202405020035.h5ad',
 '105_D03468A4C6_WT202403310067.h5ad',
 # 'A03988A1C2_WT202407161208.h5ad',
 # # 'A03591D4E5_WT2024071215074.h5ad',
 # 'A03590A3D6_WT202407192652.h5ad',
 # 'A03994F1G2_WT2024071215067.h5ad',
 #    'A03587A5C6_WT2024071215080.h5ad', # GW10
 # 'B03607C4E6_WT2024071214941.h5ad', # GW12
 #    'B03618D3F6_WT202407152793.h5ad', # GW16
]
adata = adata[adata.obs['slice_code'].isin(names)]

In [8]:
adatas = []
for i in set(adata.obs['slice_code']):
    temp = adata[adata.obs['slice_code'] == i].copy()
    temp = temp[temp.obs.sample(frac=0.1).index].copy()
    if i == '47_A03593C1F3_WT202403310068.h5ad':
        temp.obsm['align_spatial_3d'] = np.array([temp.obsm['align_spatial_2d'][:, 0],
                                                  temp.obsm['align_spatial_2d'][:, 1], 
                                                  [1187.5 for i in range(len(temp))]]).T

    # sc.pp.normalize_total(temp)
    # sc.pp.log1p(temp)
    # sc.pp.scale(temp, zero_center=False, max_value=10)
    adatas.append(temp)
adata = ad.concat(adatas)

In [9]:
adata.obsm['3d_align_spatial'] = adata.obsm['align_spatial_3d'][:, [0, 2, 1]].copy()
adata.obsm['3d_align_spatial'][:,2] = -adata.obsm['3d_align_spatial'][:,2]
adata.obsm['3d_align_spatial'] = adata.obsm['3d_align_spatial'] - adata.obsm['3d_align_spatial'].mean(axis = 0)

In [10]:
colormap.keys()

dict_keys(['T2_PRSS12', 'T6_CLSTN2', 'T3_BMP4_IGFBP5', 'T4_COL4A1_vessel', 'T10_HBZ', 'T7_SST_LHX_SIX3', 'T8_CBLN4_CBLN1', 'T1_NTS_CALB1', 'T9_CRH', 'T0_PCP4_VGF', 'T5_AQP4_gial_region'])

In [11]:
celltypes = ['T2_PRSS12', 'T6_CLSTN2', 'T3_BMP4_IGFBP5', 'T4_COL4A1_vessel', 'T10_HBZ', 'T7_SST_LHX_SIX3', 'T8_CBLN4_CBLN1', 
             'T1_NTS_CALB1', 'T9_CRH', 'T0_PCP4_VGF', 'T5_AQP4_gial_region']
for celltype in celltypes:
    colormap_c = colormap.copy()
    for c in colormap_c.keys():
        if c != celltype:
            colormap_c[c] = '#f5f5f520'
        else:
            colormap_c[c] = colormap_c[c]+'ff'
    pc, plot_cmap = st.tdr.construct_pc(adata=adata.copy(), spatial_key="3d_align_spatial", groupby="dmt_leiden_annotation_0115", key_added='region_anno', colormap = colormap_c)
    opacity = [0.1 if i != celltype else 0.8 for i in adata.obs['dmt_leiden_annotation_0115']]
    pc['region_anno_rgba'][:,3] = opacity
    st.pl.three_d_plot(
        model=pc,
        key='region_anno',
        # colormap=colormap_c,
        opacity=0.1,
        model_style="points",
        jupyter="html",
        background = '#EEEEEE',
        # cpo=[cpo]
        plotter_filename = f'3d_plot/region/{celltype}.html'
    )

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

...

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…